In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys 


PROJECT_ROOT="/home/sagemaker-user/subocol-ia"

if PROJECT_ROOT not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
from src.config import AWS_REGION,RAW_DATA_URI,S3_BUCKET,RAW_DATA_KEY
import boto3
from src.data.s3_io import get_object_metadata,load_csv_from_s3
import pandas as pd



In [5]:
df = load_csv_from_s3(
    S3_BUCKET,
    RAW_DATA_KEY
)
df.head()

,numero_aviso,fecha_creacion,tipo_carroceria,marca,linea,version,modelo,version_hechos,codigo_irs,nombre_irs,piezas_totales,piezas_cambio,estado_aviso
0,237305,2025-01-03 10:10:18.421101,pickup,VOLKSWAGEN,AMAROK,POWER PLUS 4X4 2.0 AUT,2016,maquinaria motoniveladora me choco en reversa ...,02090112nndn,parachoque del.,13,13,ENTREGADO
1,237305,2025-01-03 10:10:18.421101,pickup,VOLKSWAGEN,AMAROK,POWER PLUS 4X4 2.0 AUT,2016,maquinaria motoniveladora me choco en reversa ...,02030014nnnn,broche carroceria,13,13,ENTREGADO
2,237305,2025-01-03 10:10:18.421101,pickup,VOLKSWAGEN,AMAROK,POWER PLUS 4X4 2.0 AUT,2016,maquinaria motoniveladora me choco en reversa ...,03040008nndn,ampolleta.,13,13,ENTREGADO
3,237305,2025-01-03 10:10:18.421101,pickup,VOLKSWAGEN,AMAROK,POWER PLUS 4X4 2.0 AUT,2016,maquinaria motoniveladora me choco en reversa ...,02090129nndn,mascara del. central,13,13,ENTREGADO
4,237305,2025-01-03 10:10:18.421101,pickup,VOLKSWAGEN,AMAROK,POWER PLUS 4X4 2.0 AUT,2016,maquinaria motoniveladora me choco en reversa ...,02010229nndn,tapa de neblinero delantero izquierdo,13,13,ENTREGADO


In [6]:
print(df.shape)
print(df.columns)
print(df["numero_aviso"].nunique())

(7532, 13)
Index(['numero_aviso', 'fecha_creacion', 'tipo_carroceria', 'marca', 'linea',
       'version', 'modelo', 'version_hechos', 'codigo_irs', 'nombre_irs',
       'piezas_totales', 'piezas_cambio', 'estado_aviso'],
      dtype='object')
800


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7532 entries, 0 to 7531
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   numero_aviso     7532 non-null   int64 
 1   fecha_creacion   7532 non-null   object
 2   tipo_carroceria  2303 non-null   object
 3   marca            7532 non-null   object
 4   linea            7532 non-null   object
 5   version          7532 non-null   object
 6   modelo           7532 non-null   int64 
 7   version_hechos   7532 non-null   object
 8   codigo_irs       7485 non-null   object
 9   nombre_irs       7485 non-null   object
 10  piezas_totales   7532 non-null   int64 
 11  piezas_cambio    7532 non-null   int64 
 12  estado_aviso     7532 non-null   object
dtypes: int64(4), object(9)
memory usage: 765.1+ KB


In [8]:
print(df.isna().sum()) # missing values 

numero_aviso          0
fecha_creacion        0
tipo_carroceria    5229
marca                 0
linea                 0
version               0
modelo                0
version_hechos        0
codigo_irs           47
nombre_irs           47
piezas_totales        0
piezas_cambio         0
estado_aviso          0
dtype: int64


In [9]:
missing_percentage = (df.isna().sum()/ len(df) * 100)

print(missing_percentage)

numero_aviso        0.000000
fecha_creacion      0.000000
tipo_carroceria    69.423792
marca               0.000000
linea               0.000000
version             0.000000
modelo              0.000000
version_hechos      0.000000
codigo_irs          0.624004
nombre_irs          0.624004
piezas_totales      0.000000
piezas_cambio       0.000000
estado_aviso        0.000000
dtype: float64


In [10]:
total_claims= df["numero_aviso"].nunique()


In [11]:
claims_with_body_type=(
df.dropna(subset=["tipo_carroceria"])["numero_aviso"].nunique()
)

claims_without_body_type = total_claims - claims_with_body_type

In [12]:
print("Total claims:", total_claims)
print("Claims with body type:", claims_with_body_type)
print("Claims without body type:", claims_without_body_type)

Total claims: 800
Claims with body type: 221
Claims without body type: 579


In [13]:
claims_without_irs = (
    df[df["nombre_irs"].isna()]["numero_aviso"]
    .nunique()
)

print("Claims with at least one missing IRS part:", claims_without_irs)

Claims with at least one missing IRS part: 31


In [14]:
non_null_irs_per_claim = (
    df.groupby("numero_aviso")["nombre_irs"]
    .count()
)

print(non_null_irs_per_claim.head(10))

numero_aviso
75886    25
76221    24
77183     2
78558    16
78746     2
78926     3
78963    11
79745    76
79795     8
79950     2
Name: nombre_irs, dtype: int64


In [15]:
claims_with_all_irs_missing = (
    non_null_irs_per_claim == 0
).sum()

print("Claims with all nombre_irs missing:",
      claims_with_all_irs_missing)

Claims with all nombre_irs missing: 0


In [16]:
print(df["estado_aviso"].value_counts())

estado_aviso
OBJETADO     4720
ENTREGADO    2812
Name: count, dtype: int64


In [17]:
claim_status=(
    df[["numero_aviso","estado_aviso"]]
    .drop_duplicates()
        )

print(claim_status["estado_aviso"].value_counts())

estado_aviso
OBJETADO     500
ENTREGADO    300
Name: count, dtype: int64


In [18]:
OBJETADO  = (500 / 800) *100
ENTREGADO = 300 / 800 *100

print(OBJETADO)
print(ENTREGADO)


62.5
37.5


In [19]:
status_per_claim = (
    df.groupby("numero_aviso")["estado_aviso"]
    .nunique()
)

print(status_per_claim.value_counts())

estado_aviso
1    800
Name: count, dtype: int64


In [20]:
parts_per_claim = (
    df.groupby("numero_aviso")
    .size()
)

print(parts_per_claim.describe())

count    800.000000
mean       9.415000
std       13.573986
min        1.000000
25%        2.000000
50%        5.000000
75%       11.000000
max      110.000000
dtype: float64


In [21]:
rows_per_claim = (
    df.groupby("numero_aviso")
    .size()
)

print(rows_per_claim.sort_values(ascending=False).head(10))

numero_aviso
238588    110
185255    108
163990    107
185120    106
173619     94
205040     86
152348     82
79745      78
118814     75
211550     73
dtype: int64


In [22]:
df[df["numero_aviso"] == 238588][
    ["numero_aviso", "version_hechos", "nombre_irs", "estado_aviso"]
].head(20)

,numero_aviso,version_hechos,nombre_irs,estado_aviso
5145,238588,me dirijo desde sabana de torres con destino a...,bocel compuerta platon,OBJETADO
5146,238588,me dirijo desde sabana de torres con destino a...,bocel compuerta,OBJETADO
5147,238588,me dirijo desde sabana de torres con destino a...,panel trasero,OBJETADO
5148,238588,me dirijo desde sabana de torres con destino a...,guardapolvo metalico trasero izquierdo,OBJETADO
5149,238588,me dirijo desde sabana de torres con destino a...,NaN,OBJETADO
5150,238588,me dirijo desde sabana de torres con destino a...,caja direccion mecanica,OBJETADO
5151,238588,me dirijo desde sabana de torres con destino a...,guardapolvo plastico delantero izquierdo,OBJETADO
5152,238588,me dirijo desde sabana de torres con destino a...,guardapolvo plastico delantero derecho,OBJETADO
5153,238588,me dirijo desde sabana de torres con destino a...,sensores de aproximacion,OBJETADO
5154,238588,me dirijo desde sabana de torres con destino a...,enfocador plastico izquierdo radiador,OBJETADO


In [23]:
claim_238588 = df[df["numero_aviso"] == 238588]

print("Total rows:", len(claim_238588))

print(
    "Unique nombre_irs:",
    claim_238588["nombre_irs"].nunique()
)

Total rows: 110
Unique nombre_irs: 104


In [24]:
claim_238588["nombre_irs"].value_counts().head(10)

nombre_irs
sensores de aproximacion              3
aro antiniebla derecho                2
soporte izquierdo bomper delantero    2
soporte derecho bomper delantero      2
emblema marca                         1
puerta delantero izquierdo            1
marco frontal fibra                   1
bateria                               1
estribo izquierdo                     1
guantera                              1
Name: count, dtype: int64

In [25]:
print(df["fecha_creacion"].min())
print(df["fecha_creacion"].max())

2023-08-07 11:24:21.597000
2025-06-23 12:50:13.759403


In [26]:
fecha_dt = pd.to_datetime(
    df["fecha_creacion"],
    errors="coerce"
)

print("Invalid dates:", fecha_dt.isna().sum())
print("Earliest:", fecha_dt.min())
print("Latest:", fecha_dt.max())

Invalid dates: 0
Earliest: 2023-08-07 11:24:21.597000
Latest: 2025-06-23 12:50:13.759403


In [27]:
versions_per_claim = (
    df.groupby("numero_aviso")["version_hechos"]
    .nunique()
)

print(versions_per_claim.value_counts())

version_hechos
1    800
Name: count, dtype: int64


In [28]:
columns_to_check = [
    "fecha_creacion",
    "marca",
    "linea",
    "version",
    "modelo"
]

for column in columns_to_check:
    values_per_claim = (
        df.groupby("numero_aviso")[column]
        .nunique()
    )

    print(f"\n{column}")
    print(values_per_claim.value_counts())


fecha_creacion
fecha_creacion
1    800
Name: count, dtype: int64

marca
marca
1    800
Name: count, dtype: int64

linea
linea
1    800
Name: count, dtype: int64

version
version
1    800
Name: count, dtype: int64

modelo
modelo
1    800
Name: count, dtype: int64


In [29]:
body_types_per_claim = (
    df.groupby("numero_aviso")["tipo_carroceria"]
    .nunique()
)

print(body_types_per_claim.value_counts())

tipo_carroceria
0    579
1    221
Name: count, dtype: int64


In [30]:
print(
    df.dropna(subset=["tipo_carroceria"])
      [["numero_aviso", "tipo_carroceria"]]
      .drop_duplicates()
      ["tipo_carroceria"]
      .value_counts()
)

tipo_carroceria
camioneta     105
pickup         44
hatchback      36
sedan          31
campero         2
utilitario      2
coupe           1
Name: count, dtype: int64


In [31]:
brand_per_claim = (
    df[["numero_aviso", "marca"]]
    .drop_duplicates()
)

print(brand_per_claim["marca"].value_counts().head(15))

marca
MAZDA            111
TOYOTA            97
CHEVROLET         79
RENAULT           72
KIA               64
FORD              58
NISSAN            53
VOLKSWAGEN        37
HYUNDAI           29
HONDA             24
SUZUKI            24
BMW               20
MERCEDES-BENZ     19
SUBARU            18
JEEP              12
Name: count, dtype: int64


In [32]:
print("Unique brands:", brand_per_claim["marca"].nunique())

Unique brands: 43


In [33]:
print(
    brand_per_claim["marca"]
    .value_counts()
    .tail(10)
)

marca
SKODA               1
CITROEN             1
HAVAL               1
MAHINDRA            1
OPEL                1
DAIHATSU            1
LAND ROVER          1
GREAT WALL MOTOR    1
CUPRA               1
FOTON               1
Name: count, dtype: int64


In [34]:
line_per_claim = (
    df[["numero_aviso", "linea"]]
    .drop_duplicates()
)

print("Unique lines:", line_per_claim["linea"].nunique())

Unique lines: 360


In [35]:
print(
    line_per_claim["linea"]
    .value_counts()
    .head(15)
)

linea
2 [2]                19
CX5                  15
PICANTO [3]          14
DUSTER [2]           13
CX5 [2]              13
CX30                 13
2 [2] [FL]           11
RAV4 [5]             11
HILUX [8] [2 FL]      9
SAIL                  9
PRADO [LC 150]        8
ESCAPE [3]            8
COROLLA [12] [FL]     8
RIO                   8
3                     7
Name: count, dtype: int64


In [36]:
line_counts = line_per_claim["linea"].value_counts()

print("Lines appearing once:", (line_counts == 1).sum())
print("Lines appearing 2 times or fewer:", (line_counts <= 2).sum())
print("Lines appearing 5 times or fewer:", (line_counts <= 5).sum())

Lines appearing once: 199
Lines appearing 2 times or fewer: 266
Lines appearing 5 times or fewer: 338


In [37]:
version_per_claim = (
    df[["numero_aviso", "version"]]
    .drop_duplicates()
)

print(
    "Unique versions:",
    version_per_claim["version"].nunique()
)

Unique versions: 556


In [38]:
version_counts = (
    version_per_claim["version"]
    .value_counts()
)

print("Versions appearing once:", (version_counts == 1).sum())
print("Versions appearing 2 times or fewer:", (version_counts <= 2).sum())
print("Versions appearing 5 times or fewer:", (version_counts <= 5).sum())

Versions appearing once: 419
Versions appearing 2 times or fewer: 505
Versions appearing 5 times or fewer: 551


In [39]:
model_per_claim = (
    df[["numero_aviso", "modelo"]]
    .drop_duplicates()
)

print(model_per_claim["modelo"].describe())

count     800.000000
mean     2018.445000
std         4.226619
min      2000.000000
25%      2016.000000
50%      2019.000000
75%      2022.000000
max      2025.000000
Name: modelo, dtype: float64


In [40]:
print(
    model_per_claim["modelo"]
    .value_counts()
    .sort_index()
)

modelo
2000     1
2002     1
2004     4
2005     2
2006     3
2007     4
2008     6
2009     9
2010     8
2011    14
2012    26
2013    26
2014    32
2015    44
2016    47
2017    56
2018    64
2019    74
2020    81
2021    72
2022    91
2023    81
2024    48
2025     6
Name: count, dtype: int64


In [41]:
for column in ["piezas_totales", "piezas_cambio"]:
    values_per_claim = (
        df.groupby("numero_aviso")[column]
        .nunique()
    )

    print(f"\n{column}")
    print(values_per_claim.value_counts())


piezas_totales
piezas_totales
1    800
Name: count, dtype: int64

piezas_cambio
piezas_cambio
1    800
Name: count, dtype: int64


In [42]:
claim_parts_check = (
    df.groupby("numero_aviso")
    .agg(
        rows=("numero_aviso", "size"),
        piezas_totales=("piezas_totales", "first"),
        valid_part_names=("nombre_irs", "count")
    )
)

claim_parts_check.head(10)

,rows,piezas_totales,valid_part_names
numero_aviso,,,
75886,25,25,25
76221,26,26,24
77183,2,2,2
78558,16,16,16
78746,2,2,2
78926,3,3,3
78963,11,11,11
79745,78,78,76
79795,8,8,8


In [43]:
parts_change_check = (
    df.groupby("numero_aviso")
    .agg(
        piezas_totales=("piezas_totales", "first"),
        piezas_cambio=("piezas_cambio", "first")
    )
)

parts_change_check.head(10)

,piezas_totales,piezas_cambio
numero_aviso,,
75886,25,25
76221,26,24
77183,2,2
78558,16,16
78746,2,2
78926,3,3
78963,11,11
79745,78,76
79795,8,8


In [44]:
invalid_change_counts = (
    parts_change_check["piezas_cambio"]
    > parts_change_check["piezas_totales"]
)

print(invalid_change_counts.value_counts())

False    800
Name: count, dtype: int64


In [45]:
parts_change_check["replacement_ratio"] = (
    parts_change_check["piezas_cambio"]
    / parts_change_check["piezas_totales"]
)

print(parts_change_check["replacement_ratio"].describe())

count    800.000000
mean       0.997583
std        0.013132
min        0.900000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: replacement_ratio, dtype: float64


In [46]:
claim_eda = (
    df.groupby("numero_aviso")
    .agg(
        version_hechos=("version_hechos", "first"),
        estado_aviso=("estado_aviso", "first"),
        valid_parts=("nombre_irs", "count")
    )
)

claim_eda.head()

,version_hechos,estado_aviso,valid_parts
numero_aviso,,,
75886,baje la velocidad por el peaje y el auto me go...,OBJETADO,25
76221,ubicacion: ladera choconta hechos: iba subiend...,OBJETADO,24
77183,"ubicacion: santa ana magdalena- cicuco, boliva...",OBJETADO,2
78558,iba en via principal y en san felipe el auto d...,ENTREGADO,16
78746,"ubicacion: cra 22 cll 29, cartagena hechos: es...",OBJETADO,2


In [47]:
parts_by_claim = (
    df.groupby("numero_aviso")["nombre_irs"]
    .apply(lambda x: x.dropna().tolist())
)

In [48]:
print(type(parts_by_claim))
print(parts_by_claim.head())

<class 'pandas.core.series.Series'>
numero_aviso
75886    [piso baul, tapa baul, esquinera defensa trase...
76221    [chapa tapa baul, sellante lamina, emblema ver...
77183          [babero delantero, caja direccion mecanica]
78558    [guia defensa trasero izquierdo, soporte plast...
78746    [caja de direccion hidraulica, tornillo caja d...
Name: nombre_irs, dtype: object


In [49]:
claim_eda["parts"] = parts_by_claim

In [50]:
claim_eda.loc[75886]

version_hechos    baje la velocidad por el peaje y el auto me go...
estado_aviso                                               OBJETADO
valid_parts                                                      25
parts             [piso baul, tapa baul, esquinera defensa trase...
Name: 75886, dtype: object

In [51]:
claim_info = (
    df.groupby("numero_aviso")
    .agg(
        fecha_creacion=("fecha_creacion", "first"),
        tipo_carroceria=("tipo_carroceria", "first"),
        marca=("marca", "first"),
        linea=("linea", "first"),
        version=("version", "first"),
        modelo=("modelo", "first"),
        piezas_totales=("piezas_totales", "first"),
        piezas_cambio=("piezas_cambio", "first")
    )
)

claim_info.head()

,fecha_creacion,tipo_carroceria,marca,linea,version,modelo,piezas_totales,piezas_cambio
numero_aviso,,,,,,,,
75886,2023-08-07 11:24:21.597000,utilitario,TOYOTA,HIACE,[1],2022,25,25
76221,2023-08-09 05:32:53.474000,None,NISSAN,NP 300 FRONTIER [2],2.5L MT 2500CC TD 4X2 AA 2AB ABS,2019,26,24
77183,2023-08-15 17:06:49.222000,None,RAM,RAM [5],DT 1500 BIG HORN ETORQ CREW CAB HYBRID TP 3600...,2021,2,2
78558,2023-08-24 14:57:22.597000,sedan,HYUNDAI,ACCENT SOLARIS,SOLARIS,2019,16,16
78746,2023-08-25 14:26:39.104000,None,FORD,ESCAPE [3],TITANIUM TP 2000CC 4X4,2017,2,2


In [52]:
claims = claim_info.join(claim_eda)

In [53]:
claims.head()

,fecha_creacion,tipo_carroceria,marca,linea,version,modelo,piezas_totales,piezas_cambio,version_hechos,estado_aviso,valid_parts,parts
numero_aviso,,,,,,,,,,,,
75886,2023-08-07 11:24:21.597000,utilitario,TOYOTA,HIACE,[1],2022,25,25,baje la velocidad por el peaje y el auto me go...,OBJETADO,25,"[piso baul, tapa baul, esquinera defensa trase..."
76221,2023-08-09 05:32:53.474000,None,NISSAN,NP 300 FRONTIER [2],2.5L MT 2500CC TD 4X2 AA 2AB ABS,2019,26,24,ubicacion: ladera choconta hechos: iba subiend...,OBJETADO,24,"[chapa tapa baul, sellante lamina, emblema ver..."
77183,2023-08-15 17:06:49.222000,None,RAM,RAM [5],DT 1500 BIG HORN ETORQ CREW CAB HYBRID TP 3600...,2021,2,2,"ubicacion: santa ana magdalena- cicuco, boliva...",OBJETADO,2,"[babero delantero, caja direccion mecanica]"
78558,2023-08-24 14:57:22.597000,sedan,HYUNDAI,ACCENT SOLARIS,SOLARIS,2019,16,16,iba en via principal y en san felipe el auto d...,ENTREGADO,16,"[guia defensa trasero izquierdo, soporte plast..."
78746,2023-08-25 14:26:39.104000,None,FORD,ESCAPE [3],TITANIUM TP 2000CC 4X4,2017,2,2,"ubicacion: cra 22 cll 29, cartagena hechos: es...",OBJETADO,2,"[caja de direccion hidraulica, tornillo caja d..."


In [54]:
print(claims.shape)

(800, 12)


In [55]:
parts_length_matches = (
    claims["parts"].apply(len)
    == claims["valid_parts"]
)

print(parts_length_matches.value_counts())

True    800
Name: count, dtype: int64


In [56]:
from src.data.preprocessing import build_claim_dataset

In [57]:
claims_processed = build_claim_dataset(df)

print(claims_processed.shape)
claims_processed.head()

print(
    (
        claims_processed["parts"].apply(len)
        == claims_processed["valid_parts"]
    ).value_counts()
)

(800, 15)
True    800
Name: count, dtype: int64


In [58]:
print(claims_processed["fecha_creacion"].dtype)
print(claims_processed["fecha_creacion"].isna().sum())

datetime64[ns]
0


In [59]:
print(claims_processed["vehicle_age"].describe())

count    800.000000
mean       5.608750
std        4.265961
min        0.000000
25%        2.000000
50%        5.000000
75%        8.000000
max       24.000000
Name: vehicle_age, dtype: float64


In [60]:
print(
    (claims_processed["vehicle_age"] < 0)
    .value_counts()
)

vehicle_age
False    800
Name: count, dtype: int64


In [61]:
claims_processed[
    claims_processed["vehicle_age"] < 0
][
    ["numero_aviso", "fecha_creacion", "modelo", "vehicle_age", "marca", "linea"]
]

,numero_aviso,fecha_creacion,modelo,vehicle_age,marca,linea


In [62]:
claims_processed.isna().sum()

numero_aviso         0
fecha_creacion       0
tipo_carroceria    579
marca                0
linea                0
version              0
modelo               0
piezas_totales       0
piezas_cambio        0
version_hechos       0
estado_aviso         0
valid_parts          0
parts                0
parts_text           0
vehicle_age          0
dtype: int64

In [63]:
body_missing_by_target = pd.crosstab(
    claims_processed["tipo_carroceria"].isna(),
    claims_processed["estado_aviso"]
)

body_missing_by_target

estado_aviso,ENTREGADO,OBJETADO
tipo_carroceria,,
False,95,126
True,205,374


In [64]:
body_missing_percent = pd.crosstab(
    claims_processed["tipo_carroceria"].isna(),
    claims_processed["estado_aviso"],
    normalize="columns"
) * 100

body_missing_percent

estado_aviso,ENTREGADO,OBJETADO
tipo_carroceria,,
False,31.666667,25.2
True,68.333333,74.8


In [65]:
from src.data.split_data import split_claim_data

In [66]:
train_df, val_df, test_df = split_claim_data(claims_processed)

In [67]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (560, 15)
Validation: (120, 15)
Test: (120, 15)


In [68]:
print(train_df["estado_aviso"].value_counts(normalize=True))
print(val_df["estado_aviso"].value_counts(normalize=True))
print(test_df["estado_aviso"].value_counts(normalize=True))

estado_aviso
OBJETADO     0.625
ENTREGADO    0.375
Name: proportion, dtype: float64
estado_aviso
OBJETADO     0.625
ENTREGADO    0.375
Name: proportion, dtype: float64
estado_aviso
OBJETADO     0.625
ENTREGADO    0.375
Name: proportion, dtype: float64


In [69]:
import importlib
from src.baseline.features import (
    create_feature_transformers,
    fit_transform_train_features,
    transform_features
)

(narrative_vectorizer,parts_vectorizer,brand_encoder,numeric_scaler) = create_feature_transformers()

In [70]:
X_train = fit_transform_train_features(
    train_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)

In [71]:
X_val = transform_features(
    val_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)

X_test = transform_features(
    test_df,
    narrative_vectorizer,
    parts_vectorizer,
    brand_encoder,
    numeric_scaler
)

In [72]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (560, 5324)
X_val: (120, 5324)
X_test: (120, 5324)


In [73]:
print(
    "Brand features:",
    len(brand_encoder.categories_[0])
)

print(
    "Brands learned:",
    brand_encoder.categories_[0]
)

Brand features: 39
Brands learned: ['AUDI' 'BMW' 'BRILLIANCE' 'BYD' 'CHANGAN' 'CHERY' 'CHEVROLET' 'CITROEN'
 'DAIHATSU' 'DFSK/DFM/DFZL' 'DODGE' 'FIAT' 'FORD' 'GREAT WALL'
 'GREAT WALL MOTOR' 'HONDA' 'HYUNDAI' 'JAC' 'JEEP' 'KIA' 'LAND ROVER'
 'MAHINDRA' 'MAXUS' 'MAZDA' 'MERCEDES-BENZ' 'MG' 'MINI' 'MITSUBISHI'
 'NISSAN' 'PEUGEOT' 'RAM' 'RENAULT' 'SKODA' 'SSANGYONG' 'SUBARU' 'SUZUKI'
 'TOYOTA' 'VOLKSWAGEN' 'VOLVO']


In [74]:
from src.baseline.train import train_logistic_regression
y_train = train_df["estado_aviso"]

In [75]:
model = train_logistic_regression(
    X_train,
    y_train
)

In [76]:
print(model.classes_)

['ENTREGADO' 'OBJETADO']


In [77]:
y_val = val_df["estado_aviso"]

In [78]:
y_val_pred = model.predict(X_val)

In [79]:
print("Actual:")
print(y_val.head())

print("\nPredicted:")
print(y_val_pred[:5])

Actual:
416     OBJETADO
191     OBJETADO
196     OBJETADO
442    ENTREGADO
668     OBJETADO
Name: estado_aviso, dtype: object

Predicted:
['ENTREGADO' 'ENTREGADO' 'OBJETADO' 'OBJETADO' 'OBJETADO']


In [80]:
from src.baseline.evaluate import get_confusion_matrix

In [81]:
cm = get_confusion_matrix(
    y_val,
    y_val_pred
)

print(cm)

[[15 30]
 [10 65]]


In [82]:
from src.baseline.evaluate import evaluate_predictions
metrics = evaluate_predictions(
    y_val,
    y_val_pred
)

print(metrics)

{'precision': 0.6842105263157895, 'recall': 0.8666666666666667, 'f1': 0.7647058823529411, 'f2': 0.8227848101265823}


In [83]:
val_probabilities = model.predict_proba(X_val)

In [84]:
print(val_probabilities.shape)
print(val_probabilities[:5])

(120, 2)
[[0.55963234 0.44036766]
 [0.51450768 0.48549232]
 [0.32548949 0.67451051]
 [0.44891901 0.55108099]
 [0.29888796 0.70111204]]


In [85]:
obj_probabilities = val_probabilities[:, 1]

print(obj_probabilities.shape)
print(obj_probabilities[:5])

(120,)
[0.44036766 0.48549232 0.67451051 0.55108099 0.70111204]


In [86]:
import numpy as np

In [87]:
threshold = 0.45

y_val_pred_045 = np.where(
    obj_probabilities >= threshold,
    "OBJETADO",
    "ENTREGADO"
)

In [88]:
metrics_045 = evaluate_predictions(
    y_val,
    y_val_pred_045
)

cm_045 = get_confusion_matrix(
    y_val,
    y_val_pred_045
)

print(cm_045)

for name, value in metrics_045.items():
    print(f"{name}: {value:.4f}")

[[11 34]
 [ 7 68]]
precision: 0.6667
recall: 0.9067
f1: 0.7684
f2: 0.8458


In [89]:
thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60
]

for threshold in thresholds:
    y_pred = np.where(
        obj_probabilities >= threshold,
        "OBJETADO",
        "ENTREGADO"
    )

    metrics = evaluate_predictions(
        y_val,
        y_pred
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {metrics['precision']:.4f} | "
        f"Recall: {metrics['recall']:.4f} | "
        f"F1: {metrics['f1']:.4f} | "
        f"F2: {metrics['f2']:.4f}"
    )

Threshold: 0.30 | Precision: 0.6218 | Recall: 0.9867 | F1: 0.7629 | F2: 0.8831
Threshold: 0.35 | Precision: 0.6316 | Recall: 0.9600 | F1: 0.7619 | F2: 0.8696
Threshold: 0.40 | Precision: 0.6636 | Recall: 0.9467 | F1: 0.7802 | F2: 0.8722
Threshold: 0.45 | Precision: 0.6667 | Recall: 0.9067 | F1: 0.7684 | F2: 0.8458
Threshold: 0.50 | Precision: 0.6842 | Recall: 0.8667 | F1: 0.7647 | F2: 0.8228
Threshold: 0.55 | Precision: 0.6747 | Recall: 0.7467 | F1: 0.7089 | F2: 0.7311
Threshold: 0.60 | Precision: 0.7143 | Recall: 0.6667 | F1: 0.6897 | F2: 0.6757


In [90]:
import numpy as np
import pandas as pd

C_values = [
    0.01,
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0
]

thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60
]

results = []

for C in C_values:

    # Train one Logistic Regression model for this C
    model_c = train_logistic_regression(
        X_train,
        y_train,
        C=C
    )

    # Find which probability column corresponds to OBJETADO
    objetado_index = list(model_c.classes_).index("OBJETADO")

    # Get P(OBJETADO) for every validation claim
    obj_probabilities_c = model_c.predict_proba(
        X_val
    )[:, objetado_index]

    # Try several thresholds with the same model
    for threshold in thresholds:

        y_pred_c = np.where(
            obj_probabilities_c >= threshold,
            "OBJETADO",
            "ENTREGADO"
        )

        metrics_c = evaluate_predictions(
            y_val,
            y_pred_c
        )

        cm_c = get_confusion_matrix(
            y_val,
            y_pred_c
        )

        tn, fp, fn, tp = cm_c.ravel()

        specificity = tn / (tn + fp)
        

        results.append({
            "C": C,
            "threshold": threshold,
            "precision": metrics_c["precision"],
            "recall": metrics_c["recall"],
            "specificity": specificity,
            "f1": metrics_c["f1"],
            "f2": metrics_c["f2"],
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp
        })

In [91]:
results_df = pd.DataFrame(results)
results_df.head()

,C,threshold,precision,recall,specificity,f1,f2,TN,FP,FN,TP
0,0.01,0.30,0.625,1.0,0.0,0.769231,0.892857,0,45,0,75
1,0.01,0.35,0.625,1.0,0.0,0.769231,0.892857,0,45,0,75
2,0.01,0.40,0.625,1.0,0.0,0.769231,0.892857,0,45,0,75
3,0.01,0.45,0.625,1.0,0.0,0.769231,0.892857,0,45,0,75
4,0.01,0.50,0.625,1.0,0.0,0.769231,0.892857,0,45,0,75


In [92]:
best_by_c = results_df.loc[
    results_df.groupby("C")["f2"].idxmax()
]

best_by_c = best_by_c.sort_values("C")

best_by_c

,C,threshold,precision,recall,specificity,f1,f2,TN,FP,FN,TP
0,0.01,0.30,0.625000,1.000000,0.000000,0.769231,0.892857,0,45,0,75
10,0.10,0.45,0.630252,1.000000,0.022222,0.773196,0.894988,1,44,0,75
14,0.50,0.30,0.625000,1.000000,0.000000,0.769231,0.892857,0,45,0,75
21,1.00,0.30,0.621849,0.986667,0.000000,0.762887,0.883055,0,45,1,74
28,2.00,0.30,0.642857,0.960000,0.111111,0.770053,0.873786,5,40,3,72
35,5.00,0.30,0.651376,0.946667,0.155556,0.771739,0.867971,7,38,4,71
42,10.00,0.30,0.673267,0.906667,0.266667,0.772727,0.847880,12,33,7,68


In [93]:
reasonable_results = results_df[
    (results_df["recall"] >= 0.90)
    & (results_df["specificity"] > 0)
]

reasonable_results.sort_values(
    by="f2",
    ascending=False
).head(15)

,C,threshold,precision,recall,specificity,f1,f2,TN,FP,FN,TP
10,0.1,0.45,0.630252,1.000000,0.022222,0.773196,0.894988,1,44,0,75
11,0.1,0.50,0.640351,0.973333,0.088889,0.772487,0.881643,4,41,2,73
16,0.5,0.40,0.634783,0.973333,0.066667,0.768421,0.879518,3,42,2,73
28,2.0,0.30,0.642857,0.960000,0.111111,0.770053,0.873786,5,40,3,72
23,1.0,0.40,0.663551,0.946667,0.200000,0.780220,0.872236,9,36,4,71
29,2.0,0.35,0.657407,0.946667,0.177778,0.775956,0.870098,8,37,4,71
22,1.0,0.35,0.631579,0.960000,0.066667,0.761905,0.869565,3,42,3,72
35,5.0,0.30,0.651376,0.946667,0.155556,0.771739,0.867971,7,38,4,71
30,2.0,0.40,0.666667,0.933333,0.222222,0.777778,0.864198,10,35,5,70
17,0.5,0.45,0.648148,0.933333,0.155556,0.765027,0.857843,7,38,5,70


In [94]:
final_model = train_logistic_regression(
    X_train,
    y_train,
    C=1.0,
    class_weight="balanced"
)

In [95]:
test_probabilities = final_model.predict_proba(X_test)

In [96]:
objetado_index = list(final_model.classes_).index("OBJETADO")

obj_test_probabilities = test_probabilities[:, objetado_index]

In [97]:
final_threshold = 0.30

y_test_pred = np.where(
    obj_test_probabilities >= final_threshold,
    "OBJETADO",
    "ENTREGADO"
)

y_test = test_df["estado_aviso"]

In [98]:
test_metrics = evaluate_predictions(
    y_test,
    y_test_pred
)

test_cm = get_confusion_matrix(
    y_test,
    y_test_pred
)

print(test_cm)

for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}")

[[ 2 43]
 [ 4 71]]
precision: 0.6228
recall: 0.9467
f1: 0.7513
f2: 0.8575


In [99]:
from src.rag.prepare_documents import build_claim_document

In [100]:
sample_row=train_df.iloc[0]
sample_document=build_claim_document(sample_row)

print(sample_document)

ID del aviso: 219266

Fecha de creación: 2024-11-14 07:40:21.755000

Vehículo:
Marca: MAZDA
Línea: CX5 [2]
Versión: TOURING TP 2000CC 6AB R17 4X2
Modelo: 2020
Edad del vehículo: 4

Versión de los hechos:
caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com - vehiculos afectados: no - otros afectados: caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com

Piezas inspeccionadas:
bocel ampliacion delantero izquierdo guardafango izquierdo

Número total de piezas: 2
Número de piezas con descripción válida: 2

Decisión históric

In [101]:
from src.rag.prepare_documents import build_training_documents
training_documents = build_training_documents(train_df)
print("Number of training documents:", len(training_documents))
sample_claim_id = next(iter(training_documents))

print("Claim ID:", sample_claim_id)
print()
print(training_documents[sample_claim_id])

Number of training documents: 560
Claim ID: 219266

ID del aviso: 219266

Fecha de creación: 2024-11-14 07:40:21.755000

Vehículo:
Marca: MAZDA
Línea: CX5 [2]
Versión: TOURING TP 2000CC 6AB R17 4X2
Modelo: 2020
Edad del vehículo: 4

Versión de los hechos:
caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com - vehiculos afectados: no - otros afectados: caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com

Piezas inspeccionadas:
bocel ampliacion delantero izquierdo guardafango izquierdo

Número total de piezas: 2
Número de

In [102]:
from src.data.s3_io import upload_text_to_s3


In [ ]:

"""
from src.data.s3_io import upload_text_to_s3
sample_claim_id = 219266
sample_document = training_documents[sample_claim_id]
sample_key = f"rag/train/{sample_claim_id}.txt"

upload_text_to_s3(
    S3_BUCKET,
    sample_key,
    sample_document
) 
"""

In [ ]:
"""

metadata = get_object_metadata(
    S3_BUCKET,
    sample_key
)

print("Status:", metadata["ResponseMetadata"]["HTTPStatusCode"])
print("Size:", metadata["ContentLength"])
print("Content type:", metadata["ContentType"])

"""

In [ ]:
"""

from src.data.s3_io import upload_documents_to_s3
upload_documents_to_s3(
    S3_BUCKET,
    "rag/train",
    training_documents
)
"""

In [ ]:
"""

from src.data.s3_io import list_s3_objects
rag_objects = list_s3_objects(
    S3_BUCKET,
    "rag/train/"
)

print("Documents in S3:", len(rag_objects))
print(rag_objects[:5])

"""

In [103]:

from src.data.s3_io import upload_documents_to_s3
from src.data.s3_io import list_s3_objects
from src.rag.prepare_documents import build_claim_query

In [104]:
sample_claim = val_df.iloc[0]

sample_query = build_claim_query(sample_claim)

print(sample_query)

Vehículo:
Marca: TOYOTA
Línea: HILUX [8] [2 FL]
Versión: 2.8L TP 2800CC TD 7AB 4X4 EURO IV
Modelo: 2023
Edad del vehículo: 1

Versión de los hechos:
impacto en la parte trasera de mi vehiculo frene me di cuenta que me habia chocado una motocicleta

Piezas inspeccionadas:
bocel bomper trasero derecho puntera derecho bomper trasero sensores de aproximacion stop derecho refuerzo central bomper trasero

Número total de piezas: 5
Número de piezas con descripción válida: 5


In [105]:
print("Actual label:", sample_claim["estado_aviso"])

Actual label: OBJETADO


In [106]:
from src.config import BEDROCK_KB_ID
from src.rag.retrieve import retrieve_similar_claims

In [107]:
retrieved_claims = retrieve_similar_claims(
    query_text=sample_query,
    knowledge_base_id=BEDROCK_KB_ID,
    number_of_results=5
)

print("Number of retrieved claims:", len(retrieved_claims))

Number of retrieved claims: 5


In [108]:
for i, result in enumerate(retrieved_claims, start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", result["score"])
    print(result["content"]["text"])


--- Result 1 ---
Score: 0.6169747020093214
ID del aviso: 163302

Fecha de creación: 2024-06-06 10:23:12.636000

Vehículo:
Marca: TOYOTA
Línea: HILUX [8]
Versión: 2.7L MT 2700CC 4X4 EURO IV
Modelo: 2018
Edad del vehículo: 6

Versión de los hechos:
ubicacion:santo domingo hechos: me encontraba reversando el carro para girarlo y en el momento que freno, el freno no agarra. por lo que se estrella con un poste de luz (version de los hechos contada por persona externa, es un trabajador de la empresa) danos: hundido en persiana delantera y una fisura mas abajo, golpe en la tapa

Piezas inspeccionadas:
bocel capo soporte izquierdo bomper delantero bomper delantero soporte derecho bomper delantero bocel superior persiana rejilla bomper delantero

Número total de piezas: 6
Número de piezas con descripción válida: 6

Decisión histórica:
ENTREGADO

--- Result 2 ---
Score: 0.61652139200999
ID del aviso: 114056

Fecha de creación: 2024-01-06 13:15:13.556000

Vehículo:
Marca: TOYOTA
Línea: HILUX [8]

In [109]:
from src.rag.classify import classify_claim_with_rag

rag_prediction = classify_claim_with_rag(
    current_claim_text=sample_query,
    retrieved_claims=retrieved_claims
)

print("RAG prediction:", rag_prediction)
print("Actual label:", sample_claim["estado_aviso"])

RAG prediction: ENTREGADO
Actual label: OBJETADO


In [110]:
rag_results = []

for _, claim in val_df.head(10).iterrows():

    # 1. Build query without the true label
    query = build_claim_query(claim)

    # 2. Retrieve 5 similar TRAINING claims
    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=5
    )

    # 3. Ask Nova to classify
    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    # 4. Store result for evaluation
    rag_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

In [111]:
import pandas as pd

rag_results_df = pd.DataFrame(rag_results)

rag_results_df

,numero_aviso,actual,prediction
0,189957,OBJETADO,ENTREGADO
1,131524,OBJETADO,OBJETADO
2,133876,OBJETADO,OBJETADO
3,195151,ENTREGADO,ENTREGADO
4,243983,OBJETADO,OBJETADO
5,93574,ENTREGADO,OBJETADO
6,233671,OBJETADO,OBJETADO
7,203458,OBJETADO,ENTREGADO
8,201901,OBJETADO,OBJETADO
9,219952,OBJETADO,OBJETADO


In [112]:
rag_results_df["correct"] = (
    rag_results_df["actual"] == rag_results_df["prediction"]
)

rag_results_df

,numero_aviso,actual,prediction,correct
0,189957,OBJETADO,ENTREGADO,False
1,131524,OBJETADO,OBJETADO,True
2,133876,OBJETADO,OBJETADO,True
3,195151,ENTREGADO,ENTREGADO,True
4,243983,OBJETADO,OBJETADO,True
5,93574,ENTREGADO,OBJETADO,False
6,233671,OBJETADO,OBJETADO,True
7,203458,OBJETADO,ENTREGADO,False
8,201901,OBJETADO,OBJETADO,True
9,219952,OBJETADO,OBJETADO,True


In [113]:
print(
    "Correct predictions:",
    rag_results_df["correct"].sum(),
    "/",
    len(rag_results_df)
)

Correct predictions: 7 / 10


In [114]:
rag_results_df

,numero_aviso,actual,prediction,correct
0,189957,OBJETADO,ENTREGADO,False
1,131524,OBJETADO,OBJETADO,True
2,133876,OBJETADO,OBJETADO,True
3,195151,ENTREGADO,ENTREGADO,True
4,243983,OBJETADO,OBJETADO,True
5,93574,ENTREGADO,OBJETADO,False
6,233671,OBJETADO,OBJETADO,True
7,203458,OBJETADO,ENTREGADO,False
8,201901,OBJETADO,OBJETADO,True
9,219952,OBJETADO,OBJETADO,True


In [115]:
print("Actual labels:")
print(rag_results_df["actual"].value_counts())

print("\nPredicted labels:")
print(rag_results_df["prediction"].value_counts())

Actual labels:
actual
OBJETADO     8
ENTREGADO    2
Name: count, dtype: int64

Predicted labels:
prediction
OBJETADO     7
ENTREGADO    3
Name: count, dtype: int64


In [116]:
from src.baseline.evaluate import (
    get_confusion_matrix,
    evaluate_predictions
)

rag_cm = get_confusion_matrix(
    rag_results_df["actual"],
    rag_results_df["prediction"]
)

rag_metrics = evaluate_predictions(
    rag_results_df["actual"],
    rag_results_df["prediction"]
)

print("Confusion matrix:")
print(rag_cm)

print("\nMetrics:")
print(rag_metrics)

Confusion matrix:
[[1 1]
 [2 6]]

Metrics:
{'precision': 0.8571428571428571, 'recall': 0.75, 'f1': 0.8, 'f2': 0.7692307692307693}


In [117]:
rag_val_results = []

for i, (_, claim) in enumerate(val_df.iterrows(), start=1):

    query = build_claim_query(claim)

    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=5
    )

    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    rag_val_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

    print(f"Processed {i}/{len(val_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [118]:
rag_val_df = pd.DataFrame(rag_val_results)

In [119]:
rag_val_cm = get_confusion_matrix(
    rag_val_df["actual"],
    rag_val_df["prediction"]
)

rag_val_metrics = evaluate_predictions(
    rag_val_df["actual"],
    rag_val_df["prediction"]
)

print("Confusion matrix:")
print(rag_val_cm)

print("\nMetrics:")
print(rag_val_metrics)

Confusion matrix:
[[21 24]
 [15 60]]

Metrics:
{'precision': 0.7142857142857143, 'recall': 0.8, 'f1': 0.7547169811320755, 'f2': 0.78125}


In [120]:
rag_test_results = []

for i, (_, claim) in enumerate(test_df.iterrows(), start=1):

    query = build_claim_query(claim)

    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=5
    )

    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    rag_test_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

    print(f"Processed {i}/{len(test_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [121]:
rag_test_df = pd.DataFrame(rag_test_results)

rag_test_df.head()

,numero_aviso,actual,prediction
0,92334,OBJETADO,OBJETADO
1,206854,OBJETADO,ENTREGADO
2,237284,OBJETADO,OBJETADO
3,231175,ENTREGADO,ENTREGADO
4,79950,OBJETADO,OBJETADO


In [122]:
rag_test_cm = get_confusion_matrix(
    rag_test_df["actual"],
    rag_test_df["prediction"]
)

rag_test_metrics = evaluate_predictions(
    rag_test_df["actual"],
    rag_test_df["prediction"]
)

print("Confusion matrix:")
print(rag_test_cm)

print("\nMetrics:")
print(rag_test_metrics)

Confusion matrix:
[[23 22]
 [12 63]]

Metrics:
{'precision': 0.7411764705882353, 'recall': 0.84, 'f1': 0.7875, 'f2': 0.8181818181818182}


In [124]:
from src.rag.retrieve import (
    retrieve_similar_claims,
    select_balanced_claims
)

In [125]:
from src.rag.retrieve import (
    retrieve_similar_claims,
    select_balanced_claims
)

In [126]:
balanced_claims = select_balanced_claims(
    retrieved_claims,
    per_class=3
)

print("Retrieved:", len(retrieved_claims))
print("Selected:", len(balanced_claims))

Retrieved: 5
Selected: 5


In [127]:
from src.rag.retrieve import get_historical_label

for i, result in enumerate(balanced_claims, start=1):
    print(
        i,
        get_historical_label(result),
        result["score"]
    )

1 ENTREGADO 0.6169747020093214
2 OBJETADO 0.61652139200999
3 OBJETADO 0.6095236102254054
4 ENTREGADO 0.6080549187175103
5 ENTREGADO 0.594615940180358


In [128]:
balanced_prediction = classify_claim_with_rag(
    current_claim_text=sample_query,
    retrieved_claims=balanced_claims
)

print("Balanced RAG prediction:", balanced_prediction)
print("Actual label:", sample_claim["estado_aviso"])

Balanced RAG prediction: ENTREGADO
Actual label: OBJETADO


In [129]:
print("Total retrieved:", len(retrieved_claims))

for i, result in enumerate(retrieved_claims, start=1):
    print(
        i,
        get_historical_label(result),
        result["score"]
    )

Total retrieved: 5
1 ENTREGADO 0.6169747020093214
2 OBJETADO 0.61652139200999
3 OBJETADO 0.6095236102254054
4 ENTREGADO 0.6080549187175103
5 ENTREGADO 0.594615940180358


In [130]:
from collections import Counter

labels = [
    get_historical_label(result)
    for result in retrieved_claims
]

print(Counter(labels))

Counter({'ENTREGADO': 3, 'OBJETADO': 2})


In [140]:
retrieved_claims = retrieve_similar_claims(
    query_text=sample_query,
    knowledge_base_id=BEDROCK_KB_ID,
    number_of_results=30
)

print("Total retrieved:", len(retrieved_claims))

Total retrieved: 30


In [141]:
for i, result in enumerate(retrieved_claims, start=1):
    print(
        i,
        get_historical_label(result),
        result["score"]
    )

1 ENTREGADO 0.6169747020093214
2 OBJETADO 0.61652139200999
3 OBJETADO 0.6095236102254054
4 ENTREGADO 0.6080549187175103
5 ENTREGADO 0.594615940180358
6 ENTREGADO 0.592843893530325
7 ENTREGADO 0.5849151906554229
8 OBJETADO 0.5842398243479482
9 ENTREGADO 0.5807638346327686
10 OBJETADO 0.5787291007647977
11 ENTREGADO 0.5714256131072319
12 ENTREGADO 0.5694923446916916
13 OBJETADO 0.5691261405698027
14 OBJETADO 0.5669189784641482
15 ENTREGADO 0.5640898634655833
16 OBJETADO 0.5632077499503922
17 ENTREGADO 0.5628596080411161
18 OBJETADO 0.5520018751436989
19 OBJETADO 0.5506723650254651
20 OBJETADO 0.5491134176101551
21 ENTREGADO 0.5480166231419832
22 OBJETADO 0.5469719881049387
23 OBJETADO 0.5465062099661748
24 ENTREGADO 0.5451367374897733
25 OBJETADO 0.5414638072719619
26 OBJETADO 0.5413027699684173
27 OBJETADO 0.5410794539647517
28 ENTREGADO 0.5410293588734686
29 OBJETADO 0.5394968893711509
30 OBJETADO 0.5390473177014087


In [142]:
from collections import Counter

labels = [
    get_historical_label(result)
    for result in retrieved_claims
]

print(Counter(labels))

Counter({'OBJETADO': 17, 'ENTREGADO': 13})


In [143]:
balanced_claims = select_balanced_claims(
    retrieved_claims,
    per_class=3
)

print("Balanced selected:", len(balanced_claims))

for i, result in enumerate(balanced_claims, start=1):
    print(
        i,
        get_historical_label(result),
        result["score"]
    )

Balanced selected: 6
1 ENTREGADO 0.6169747020093214
2 OBJETADO 0.61652139200999
3 OBJETADO 0.6095236102254054
4 ENTREGADO 0.6080549187175103
5 ENTREGADO 0.594615940180358
6 OBJETADO 0.5842398243479482


In [135]:
balanced_prediction = classify_claim_with_rag(
    current_claim_text=sample_query,
    retrieved_claims=balanced_claims
)

print("Balanced RAG prediction:", balanced_prediction)
print("Actual label:", sample_claim["estado_aviso"])

Balanced RAG prediction: OBJETADO
Actual label: OBJETADO


In [136]:
balanced_rag_val_results = []

for i, (_, claim) in enumerate(val_df.iterrows(), start=1):

    # Build query without the real label
    query = build_claim_query(claim)

    # Retrieve a larger candidate pool
    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=10
    )

    # Select up to 3 OBJETADO + 3 ENTREGADO
    balanced = select_balanced_claims(
        retrieved,
        per_class=3
    )

    # Classify using only the balanced historical examples
    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=balanced
    )

    balanced_rag_val_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction,
        "retrieved_candidates": len(retrieved),
        "selected_examples": len(balanced)
    })

    print(f"Processed {i}/{len(val_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [137]:
balanced_rag_val_df = pd.DataFrame(
    balanced_rag_val_results
)

balanced_rag_val_df.head()

,numero_aviso,actual,prediction,retrieved_candidates,selected_examples
0,189957,OBJETADO,OBJETADO,10,6
1,131524,OBJETADO,OBJETADO,10,6
2,133876,OBJETADO,OBJETADO,10,6
3,195151,ENTREGADO,OBJETADO,10,6
4,243983,OBJETADO,OBJETADO,10,6


In [138]:
balanced_rag_val_df["selected_examples"].value_counts().sort_index()

selected_examples
3     1
4     4
5    21
6    94
Name: count, dtype: int64

In [139]:
balanced_rag_val_cm = get_confusion_matrix(
    balanced_rag_val_df["actual"],
    balanced_rag_val_df["prediction"]
)

balanced_rag_val_metrics = evaluate_predictions(
    balanced_rag_val_df["actual"],
    balanced_rag_val_df["prediction"]
)

print("Confusion matrix:")
print(balanced_rag_val_cm)

print("\nMetrics:")
print(balanced_rag_val_metrics)

Confusion matrix:
[[19 26]
 [17 58]]

Metrics:
{'precision': 0.6904761904761905, 'recall': 0.7733333333333333, 'f1': 0.7295597484276729, 'f2': 0.7552083333333334}


In [144]:
def evaluate_rag_with_k(val_df, k):

    results = []

    for i, (_, claim) in enumerate(val_df.iterrows(), start=1):

        query = build_claim_query(claim)

        retrieved = retrieve_similar_claims(
            query_text=query,
            knowledge_base_id=BEDROCK_KB_ID,
            number_of_results=k
        )

        prediction = classify_claim_with_rag(
            current_claim_text=query,
            retrieved_claims=retrieved
        )

        results.append({
            "numero_aviso": claim["numero_aviso"],
            "actual": claim["estado_aviso"],
            "prediction": prediction
        })

        print(f"K={k} | Processed {i}/{len(val_df)}")

    results_df = pd.DataFrame(results)

    cm = get_confusion_matrix(
        results_df["actual"],
        results_df["prediction"]
    )

    metrics = evaluate_predictions(
        results_df["actual"],
        results_df["prediction"]
    )

    return results_df, cm, metrics

In [145]:
rag_val_k10_df, cm_k10, metrics_k10 = evaluate_rag_with_k(
    val_df,
    k=10
)

print(cm_k10)
print(metrics_k10)

K=10 | Processed 1/120
K=10 | Processed 2/120
K=10 | Processed 3/120
K=10 | Processed 4/120
K=10 | Processed 5/120
K=10 | Processed 6/120
K=10 | Processed 7/120
K=10 | Processed 8/120
K=10 | Processed 9/120
K=10 | Processed 10/120
K=10 | Processed 11/120
K=10 | Processed 12/120
K=10 | Processed 13/120
K=10 | Processed 14/120
K=10 | Processed 15/120
K=10 | Processed 16/120
K=10 | Processed 17/120
K=10 | Processed 18/120
K=10 | Processed 19/120
K=10 | Processed 20/120
K=10 | Processed 21/120
K=10 | Processed 22/120
K=10 | Processed 23/120
K=10 | Processed 24/120
K=10 | Processed 25/120
K=10 | Processed 26/120
K=10 | Processed 27/120
K=10 | Processed 28/120
K=10 | Processed 29/120
K=10 | Processed 30/120
K=10 | Processed 31/120
K=10 | Processed 32/120
K=10 | Processed 33/120
K=10 | Processed 34/120
K=10 | Processed 35/120
K=10 | Processed 36/120
K=10 | Processed 37/120
K=10 | Processed 38/120
K=10 | Processed 39/120
K=10 | Processed 40/120
K=10 | Processed 41/120
K=10 | Processed 42/120
K

In [146]:
rag_val_k20_df, cm_k20, metrics_k20 = evaluate_rag_with_k(
    val_df,
    k=20
)

print(cm_k20)
print(metrics_k20)

K=20 | Processed 1/120
K=20 | Processed 2/120
K=20 | Processed 3/120
K=20 | Processed 4/120
K=20 | Processed 5/120
K=20 | Processed 6/120
K=20 | Processed 7/120
K=20 | Processed 8/120
K=20 | Processed 9/120
K=20 | Processed 10/120
K=20 | Processed 11/120
K=20 | Processed 12/120
K=20 | Processed 13/120
K=20 | Processed 14/120
K=20 | Processed 15/120
K=20 | Processed 16/120
K=20 | Processed 17/120
K=20 | Processed 18/120
K=20 | Processed 19/120
K=20 | Processed 20/120
K=20 | Processed 21/120
K=20 | Processed 22/120
K=20 | Processed 23/120
K=20 | Processed 24/120
K=20 | Processed 25/120
K=20 | Processed 26/120
K=20 | Processed 27/120
K=20 | Processed 28/120
K=20 | Processed 29/120
K=20 | Processed 30/120
K=20 | Processed 31/120
K=20 | Processed 32/120
K=20 | Processed 33/120
K=20 | Processed 34/120
K=20 | Processed 35/120
K=20 | Processed 36/120
K=20 | Processed 37/120
K=20 | Processed 38/120
K=20 | Processed 39/120
K=20 | Processed 40/120
K=20 | Processed 41/120
K=20 | Processed 42/120
K

In [147]:
false_negatives = rag_val_df[
    (rag_val_df["actual"] == "OBJETADO") &
    (rag_val_df["prediction"] == "ENTREGADO")
]

print("False negatives:", len(false_negatives))

false_negatives

False negatives: 15


,numero_aviso,actual,prediction
0,189957,OBJETADO,ENTREGADO
7,203458,OBJETADO,ENTREGADO
13,93722,OBJETADO,ENTREGADO
17,140869,OBJETADO,ENTREGADO
23,156663,OBJETADO,ENTREGADO
30,217988,OBJETADO,ENTREGADO
34,223265,OBJETADO,ENTREGADO
41,224467,OBJETADO,ENTREGADO
50,84374,OBJETADO,ENTREGADO
58,150306,OBJETADO,ENTREGADO


In [148]:
false_negative_details = val_df[
    val_df["numero_aviso"].isin(
        false_negatives["numero_aviso"]
    )
][[
    "numero_aviso",
    "marca",
    "linea",
    "version_hechos",
    "parts_text",
    "estado_aviso"
]]

false_negative_details

,numero_aviso,marca,linea,version_hechos,parts_text,estado_aviso
416,189957,TOYOTA,HILUX [8] [2 FL],impacto en la parte trasera de mi vehiculo fre...,bocel bomper trasero derecho puntera derecho b...,OBJETADO
483,203458,MAZDA,3 [3],dano mecanico / motor en la correa se escucha ...,base amortiguador delantero derecho caja de di...,OBJETADO
62,93722,MAZDA,2 [2],hechos: iba pasando por un sitio que tenia un ...,silicona tornillo broches bomper delantero ara...,OBJETADO
224,140869,KIA,RIO,ubicacion: salgar antioquia. hechos: veniamos ...,liquido frenos liquido refrigerante por galon ...,OBJETADO
268,156663,RENAULT,STEPWAY [2] [FL],ok administrativo: juanprpu:fri may 17 08:27:2...,antena radio electrica vidrio panoramico delan...,OBJETADO
542,217988,VOLKSWAGEN,GOL [7] [FL],estaba saliendo del club campestre en la cra 1...,soporte izquierdo bomper delantero bocel infer...,OBJETADO
569,223265,MAZDA,2 [2] [FL],ubicacion: centro comerial el tesoro hechos: y...,broche tapizado puerta puerta delantero derech...,OBJETADO
577,224467,DFSK/DFM/DFZL,560,detenido en su semaforo cuando un tercero me i...,luz antiniebla izquierdo broches parachoque tr...,OBJETADO
21,84374,HONDA,CRV [5][FL],ubicacion: carrera 26a 10a 66 hechos: mi vehic...,moldura puerta trasero izquierdo,OBJETADO
253,150306,TOYOTA,RAV4 [5],venia transitando sobre la avenida los balsos ...,bomper trasero,OBJETADO


In [149]:
pd.set_option("display.max_colwidth", None)

In [150]:
false_negative_details

,numero_aviso,marca,linea,version_hechos,parts_text,estado_aviso
416,189957,TOYOTA,HILUX [8] [2 FL],impacto en la parte trasera de mi vehiculo frene me di cuenta que me habia chocado una motocicleta,bocel bomper trasero derecho puntera derecho bomper trasero sensores de aproximacion stop derecho refuerzo central bomper trasero,OBJETADO
483,203458,MAZDA,3 [3],dano mecanico / motor en la correa se escucha un ruido en la parte de adelante y como si se estuviera golpeando,base amortiguador delantero derecho caja de direccion hidraulica base amortiguador delantero izquierdo tensor barra estabilizadora delantero derecho tensor barra estabilizadora delantero izquierdo amortiguador delantero derecho amortiguador delantero izquierdo,OBJETADO
62,93722,MAZDA,2 [2],"hechos: iba pasando por un sitio que tenia un charco de agua grande, yo pase despacio pero se le entro agua a la parte de abajo, este paso despego el bomper, empezo arrastrar el carro se apago danos: bomper, babero, ruido en el motor cuenta con soportes que demuestren la culpabilidad del otro: no",silicona tornillo broches bomper delantero arandela bocel inferior bomper delantero babero delantero aceite 1/4 filtro de aceite reten cigueñal tensor cadena de reparticion empaque tapa valvulas solenoide freno motor,OBJETADO
224,140869,KIA,RIO,"ubicacion: salgar antioquia. hechos: veniamos en la carretera, un camion invade nuestro carril, por esquivarle nos caimos a una zanja, empezo a sonar de forma extrana y nos dimos cuenta que tenia fugas de aceite. danos: por debajo se rompieron varias piezas y esta botando aceite por todos lados.",liquido frenos liquido refrigerante por galon silicona motor 7/8 filtro de aceite aceite 1/4 babero delantero empaque multiple de admision guardapolvo plastico delantero izquierdo guardapolvo plastico delantero derecho,OBJETADO
268,156663,RENAULT,STEPWAY [2] [FL],ok administrativo: juanprpu:fri may 17 08:27:28 cot 2024<br>,antena radio electrica vidrio panoramico delantero pegante vidrio,OBJETADO
542,217988,VOLKSWAGEN,GOL [7] [FL],estaba saliendo del club campestre en la cra 102 con calle 11 en el cruce a mano izquierda frente muy tarde y la moto no alcanzo a frenar y se dio de frente contra mi carro - vehiculos afectados: no - otros afectados: no,soporte izquierdo bomper delantero bocel inferior bomper delantero bomper delantero soporte derecho bomper delantero,OBJETADO
569,223265,MAZDA,2 [2] [FL],ubicacion: centro comerial el tesoro hechos: yo iba entrando a un parqueadero el jueves 14 de noviembre se le dio al carro al ladoderecho y se afecto la puerta del coopiloto danos: puerta del coopiloto celular:3163399258 e-mail:norava05@gmail.com,broche tapizado puerta puerta delantero derecho cinta anterior puerta delantero derecho cinta decorativa trasero estribo derecho,OBJETADO
577,224467,DFSK/DFM/DFZL,560,"detenido en su semaforo cuando un tercero me impacta por detras , afectando port alon",luz antiniebla izquierdo broches parachoque tras. parachoque tras. tapa maleta refuerzo parachoque tras sup,OBJETADO
21,84374,HONDA,CRV [5][FL],ubicacion: carrera 26a 10a 66 hechos: mi vehiculo estaba estacionado y mi vecino retrocediendo colisiono mi vehiculo en la puerta trasera danos: puerta trasera datos del tercero placa : mnl441 nombre: libia gaviria tel: 3137951606,moldura puerta trasero izquierdo,OBJETADO
253,150306,TOYOTA,RAV4 [5],venia transitando sobre la avenida los balsos sentido oriente occidente al llegar a la se?al de pare antes del puente de la aguacatala me detuve y un vehiculo que venia detras no logr? frenar y me impact? en la parte trasera. - vehiculos afectados: no - otros afectados: no,bomper trasero,OBJETADO


In [151]:
from src.data.preprocessing import build_claim_dataset

claims_processed = build_claim_dataset(df)

In [152]:
claims_processed[
    [
        "numero_aviso",
        "version_hechos",
        "reported_zones",
        "parts_text",
        "parts_zones"
    ]
].head(20)

,numero_aviso,version_hechos,reported_zones,parts_text,parts_zones
0,75886,baje la velocidad por el peaje y el auto me golpea por atras.,[UNKNOWN],piso baul tapa baul esquinera defensa trasero derecho esquinera defensa trasero izquierdo laton trasero brazo refuerzo izquierdo soporte defensa trasero brazo refuerzo derecho soporte defensa trasero defensa trasero caucho tapa baul caucho parabrisas trasero soporte central defensa trasero clips defensa calcomania porton cerradura tapa baul filler lampara trasero derecho guardafango trasero derecho guia lampara trasero derecho guia lampara trasero izquierdo tapizado piso baul lampara trasero derecho refuerzo central defensa trasero vidrio parabrisas trasero base portalampara trasero derecho base portalampara trasero izquierdo tapiz inferior tapa baul,"[REAR, LEFT, RIGHT]"
1,76221,"ubicacion: ladera choconta hechos: iba subiendo una ladera, senti un golpe en la parte baja del vehiculo y el vehiculo se apago y no volvio a encender danos: chasis partido",[UNKNOWN],chapa tapa baul sellante lamina emblema version tope compuerta panel trasero manija apertura exterior tapa baul piso habitaculo pasajeros cantonera cierre compuerta panel trasero costado izquierdo salpicadera trasero izquierdo emblema version salpicadera trasero derecho stop derecho compuerta stop izquierdo emblema linea calcomania costado izquierdo costado derecho barra antivuelco bomper trasero emblema marca bisagra derecho tapa baul calcomania costado derecho,"[REAR, LEFT, RIGHT]"
2,77183,"ubicacion: santa ana magdalena- cicuco, bolivar. viajabamos hacia cartagena. hechos: cuando veniamos de regreso se empezo a sentir el timon duro y por miedo a algun accidente, buscamos una grua en el pueblo mas cercano y movilizamos el vehiculo hasta un taller en cartagena, pero como ya era tarde por lo cual la cita nos la dieron para el dia lunes, lo llevamos ya asi y ya el timon al girarse generaba un ruido, en el taller al que lo llevamos indicaron que se causo por un golpe que tuvo el vehiculo. danos: dano en la caja de direccion reportan el el taller y hacer el cambio. del mismo.",[UNKNOWN],babero delantero caja direccion mecanica,[FRONT]
3,78558,"iba en via principal y en san felipe el auto de adelante freno, logro frenar pero el auto que vania atras me colisiono",[UNKNOWN],guia defensa trasero izquierdo soporte plastico derecho defensa delantero defensa trasero piso baul refuerzo central defensa trasero laton trasero brazo refuerzo derecho soporte defensa trasero brazo refuerzo izquierdo soporte defensa trasero lampara delantero derecho parrilla guia defensa trasero derecho tapa motor defensa delantero luz muerta izquierdo defensa trasero marco radiador soporte plastico derecho defensa delantero,"[FRONT, REAR, LEFT, RIGHT]"
4,78746,"ubicacion: cra 22 cll 29, cartagena hechos: estaba lloviendo muy fuerte y pase por un lugar con bastante agua y al direccion se puso muy dura y el carro se me apago. danos: se le cayo el protector del vh",[UNKNOWN],caja de direccion hidraulica tornillo caja de direccion,[UNKNOWN]
5,78926,estaba dando reversa y cae una rama de un ?rbol encima del vehiculo - vehiculos afectados: no - otros afectados: no,[UNKNOWN],tercer stop pegante vidrio vidrio panoramico trasero,[REAR]
6,78963,"via de pereira hacia risaralda caldas. iba por la via hacia risaralda caldas en un momento me encuentro con un hueco en la via reacciono y al maniobrar le pego por debajo al auto con un separador de cemento ocasionando los siguientes danos. danos: danos en el amortiguador, se danaron las tijeras, terminales, bujes y danos internos a determinar",[UNDERBODY],amortiguador delantero izquierdo salpicadera delantero izquierdo salpicadera delantero derecho mangueta delantero izquierdo base amortiguador delantero izquierdo rodamiento delantero izquierdo tijera delantero inferior izquierdo broches bomper delantero caja direccion mecanica protector plastico motor broches bomper delantero,"[FRONT, LEFT, RIGHT]"
7,79745,"hechos: se suben 2

In [153]:
claims_processed[
    claims_processed["numero_aviso"] == 189957
][
    [
        "numero_aviso",
        "version_hechos",
        "reported_zones",
        "parts_text",
        "parts_zones",
        "estado_aviso"
    ]
]

,numero_aviso,version_hechos,reported_zones,parts_text,parts_zones,estado_aviso
416,189957,impacto en la parte trasera de mi vehiculo frene me di cuenta que me habia chocado una motocicleta,[REAR],bocel bomper trasero derecho puntera derecho bomper trasero sensores de aproximacion stop derecho refuerzo central bomper trasero,"[REAR, RIGHT]",OBJETADO


In [157]:
claims_processed = build_claim_dataset(df)
claims_processed.shape

(800, 18)

In [158]:
claims_processed["zone_consistency"].value_counts()

zone_consistency
YES        459
UNKNOWN    292
NO          49
Name: count, dtype: int64

In [160]:
pd.crosstab(
    claims_processed["zone_consistency"],
    claims_processed["estado_aviso"]
)

estado_aviso,ENTREGADO,OBJETADO
zone_consistency,,
NO,15,34
UNKNOWN,92,200
YES,193,266


In [161]:
pd.crosstab(
    claims_processed["zone_consistency"],
    claims_processed["estado_aviso"],
    normalize="index"
)

estado_aviso,ENTREGADO,OBJETADO
zone_consistency,,
NO,0.306122,0.693878
UNKNOWN,0.315068,0.684932
YES,0.420479,0.579521


In [162]:
claims_processed = build_claim_dataset(df)

claims_processed["event_type"].value_counts()

event_type
OTHER                   315
COLLISION               191
PARKED_VEHICLE          125
MOTORCYCLE_COLLISION     82
MECHANICAL               39
THEFT_VANDALISM          19
WATER                    15
ADMINISTRATIVE           14
Name: count, dtype: int64

In [163]:
pd.crosstab(
    claims_processed["event_type"],
    claims_processed["estado_aviso"]
)

estado_aviso,ENTREGADO,OBJETADO
event_type,,
ADMINISTRATIVE,4,10
COLLISION,90,101
MECHANICAL,7,32
MOTORCYCLE_COLLISION,39,43
OTHER,108,207
PARKED_VEHICLE,42,83
THEFT_VANDALISM,7,12
WATER,3,12


In [164]:
pd.crosstab(
    claims_processed["event_type"],
    claims_processed["estado_aviso"],
    normalize="index"
).round(3)

estado_aviso,ENTREGADO,OBJETADO
event_type,,
ADMINISTRATIVE,0.286,0.714
COLLISION,0.471,0.529
MECHANICAL,0.179,0.821
MOTORCYCLE_COLLISION,0.476,0.524
OTHER,0.343,0.657
PARKED_VEHICLE,0.336,0.664
THEFT_VANDALISM,0.368,0.632
WATER,0.200,0.800


In [165]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [166]:
from src.data.preprocessing import build_claim_dataset
from src.data.split_data import split_claim_data

from src.baseline.features import (
    create_feature_transformers,
    fit_transform_train_features,
    transform_features
)

from src.baseline.train import train_logistic_regression

from src.baseline.evaluate import (
    get_confusion_matrix,
    evaluate_predictions
)

import numpy as np

In [167]:
claims_processed = build_claim_dataset(df)

print("Shape:", claims_processed.shape)

claims_processed[
    [
        "numero_aviso",
        "event_type",
        "reported_zones",
        "parts_zones",
        "zone_consistency",
        "estado_aviso"
    ]
].head()

Shape: (800, 19)


,numero_aviso,event_type,reported_zones,parts_zones,zone_consistency,estado_aviso
0,75886,OTHER,[UNKNOWN],"[REAR, LEFT, RIGHT]",UNKNOWN,OBJETADO
1,76221,COLLISION,[UNKNOWN],"[REAR, LEFT, RIGHT]",UNKNOWN,OBJETADO
2,77183,OTHER,[UNKNOWN],[FRONT],UNKNOWN,OBJETADO
3,78558,COLLISION,[UNKNOWN],"[FRONT, REAR, LEFT, RIGHT]",UNKNOWN,ENTREGADO
4,78746,WATER,[UNKNOWN],[UNKNOWN],UNKNOWN,OBJETADO


In [168]:
train_df, val_df, test_df = split_claim_data(
    claims_processed
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (560, 19)
Validation: (120, 19)
Test: (120, 19)


In [169]:
print("Train:")
print(train_df["estado_aviso"].value_counts())

print("\nValidation:")
print(val_df["estado_aviso"].value_counts())

Train:
estado_aviso
OBJETADO     350
ENTREGADO    210
Name: count, dtype: int64

Validation:
estado_aviso
OBJETADO     75
ENTREGADO    45
Name: count, dtype: int64


In [170]:
(
    narrative_vectorizer,
    parts_vectorizer,
    categorical_encoder,
    numeric_scaler
) = create_feature_transformers()

In [171]:
X_train = fit_transform_train_features(
    train_df,
    narrative_vectorizer,
    parts_vectorizer,
    categorical_encoder,
    numeric_scaler
)

In [172]:
X_val = transform_features(
    val_df,
    narrative_vectorizer,
    parts_vectorizer,
    categorical_encoder,
    numeric_scaler
)

In [173]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (560, 5335)
X_val: (120, 5335)


In [174]:
y_train = train_df["estado_aviso"]
y_val = val_df["estado_aviso"]

In [175]:
print(y_train.value_counts())
print()
print(y_val.value_counts())

estado_aviso
OBJETADO     350
ENTREGADO    210
Name: count, dtype: int64

estado_aviso
OBJETADO     75
ENTREGADO    45
Name: count, dtype: int64


In [176]:
model_enriched = train_logistic_regression(
    X_train,
    y_train,
    C=1.0,
    class_weight="balanced"
)

In [177]:
objetado_index = list(
    model_enriched.classes_
).index("OBJETADO")

val_probabilities = model_enriched.predict_proba(
    X_val
)

obj_val_probabilities = val_probabilities[
    :, objetado_index
]

In [178]:
FINAL_THRESHOLD = 0.30

In [179]:
ENRICHED_THRESHOLD = 0.30

y_val_pred_enriched = np.where(
    obj_val_probabilities >= ENRICHED_THRESHOLD,
    "OBJETADO",
    "ENTREGADO"
)

In [180]:
enriched_cm = get_confusion_matrix(
    y_val,
    y_val_pred_enriched
)

enriched_metrics = evaluate_predictions(
    y_val,
    y_val_pred_enriched
)

print("Confusion matrix:")
print(enriched_cm)

print("\nMetrics:")
print(enriched_metrics)

Confusion matrix:
[[ 7 38]
 [ 5 70]]

Metrics:
{'precision': 0.6481481481481481, 'recall': 0.9333333333333333, 'f1': 0.7650273224043715, 'f2': 0.8578431372549019}


In [181]:
tn, fp, fn, tp = enriched_cm.ravel()

specificity = tn / (tn + fp)

print("\nSpecificity:", specificity)


Specificity: 0.15555555555555556


In [182]:
threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):

    predictions = np.where(
        obj_val_probabilities >= threshold,
        "OBJETADO",
        "ENTREGADO"
    )

    cm = get_confusion_matrix(
        y_val,
        predictions
    )

    metrics = evaluate_predictions(
        y_val,
        predictions
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp)

    accuracy = (tp + tn) / len(y_val)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "f2": metrics["f2"],
        "specificity": specificity,
        "accuracy": accuracy,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

In [183]:
threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

,threshold,precision,recall,f1,f2,specificity,accuracy,TN,FP,FN,TP
0,0.30,0.648148,0.933333,0.765027,0.857843,0.155556,0.641667,7,38,5,70
1,0.35,0.660000,0.880000,0.754286,0.825000,0.244444,0.641667,11,34,9,66
2,0.40,0.674157,0.800000,0.731707,0.771208,0.355556,0.633333,16,29,15,60
3,0.45,0.692308,0.720000,0.705882,0.714286,0.466667,0.625000,21,24,21,54
4,0.50,0.758621,0.586667,0.661654,0.614525,0.688889,0.625000,31,14,31,44
5,0.55,0.759259,0.546667,0.635659,0.579096,0.711111,0.608333,32,13,34,41
6,0.60,0.756098,0.413333,0.534483,0.454545,0.777778,0.550000,35,10,44,31
7,0.65,0.806452,0.333333,0.471698,0.377644,0.866667,0.533333,39,6,50,25
8,0.70,0.857143,0.240000,0.375000,0.280374,0.933333,0.500000,42,3,57,18


In [184]:
threshold_df[
    threshold_df["recall"] >= 0.90
].sort_values(
    by="specificity",
    ascending=False
)

,threshold,precision,recall,f1,f2,specificity,accuracy,TN,FP,FN,TP
0,0.3,0.648148,0.933333,0.765027,0.857843,0.155556,0.641667,7,38,5,70


In [185]:
from src.rag.prepare_documents import (
    build_training_documents,
    build_claim_query
)

In [186]:
training_documents = build_training_documents(
    train_df
)

print("Training documents:", len(training_documents))

Training documents: 560


In [187]:
sample_id = next(iter(training_documents))

print(training_documents[sample_id])

ID del aviso: 219266

Fecha de creación: 2024-11-14 07:40:21.755000

Vehículo:
Marca: MAZDA
Línea: CX5 [2]
Versión: TOURING TP 2000CC 6AB R17 4X2
Modelo: 2020
Edad del vehículo: 4

Tipo de evento:
PARKED_VEHICLE

Zona reportada del impacto:
UNKNOWN

Zona de las piezas inspeccionadas:
FRONT, LEFT

Consistencia de zonas:
UNKNOWN

Versión de los hechos:
caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com - vehiculos afectados: no - otros afectados: caso ley 2251 asegurado tiene su vh estacionado y el tercero reversando sin preocupacion lo colisiona. las partes se retiran del lugar y cliente solicita asistencia virtual tercero responsable asegurado con mapfre nnu185 robeiro bastidas sanchez cc 70878281 3108445361 almiro.o@hotmail.com

Piezas inspeccionadas

In [188]:
from src.data.s3_io import (
    upload_documents_to_s3,
    list_s3_objects
)

from src.config import S3_BUCKET

In [189]:
upload_documents_to_s3(
    bucket=S3_BUCKET,
    prefix="rag/train",
    documents=training_documents
)

In [190]:
rag_files = list_s3_objects(
    S3_BUCKET,
    "rag/train/"
)

print("Documents in S3:", len(rag_files))

Documents in S3: 560


In [191]:
enriched_rag_val_results = []

for i, (_, claim) in enumerate(val_df.iterrows(), start=1):

    query = build_claim_query(claim)

    retrieved = retrieve_similar_claims(
        query_text=query,
        knowledge_base_id=BEDROCK_KB_ID,
        number_of_results=5
    )

    prediction = classify_claim_with_rag(
        current_claim_text=query,
        retrieved_claims=retrieved
    )

    enriched_rag_val_results.append({
        "numero_aviso": claim["numero_aviso"],
        "actual": claim["estado_aviso"],
        "prediction": prediction
    })

    print(f"Processed {i}/{len(val_df)}")

Processed 1/120
Processed 2/120
Processed 3/120
Processed 4/120
Processed 5/120
Processed 6/120
Processed 7/120
Processed 8/120
Processed 9/120
Processed 10/120
Processed 11/120
Processed 12/120
Processed 13/120
Processed 14/120
Processed 15/120
Processed 16/120
Processed 17/120
Processed 18/120
Processed 19/120
Processed 20/120
Processed 21/120
Processed 22/120
Processed 23/120
Processed 24/120
Processed 25/120
Processed 26/120
Processed 27/120
Processed 28/120
Processed 29/120
Processed 30/120
Processed 31/120
Processed 32/120
Processed 33/120
Processed 34/120
Processed 35/120
Processed 36/120
Processed 37/120
Processed 38/120
Processed 39/120
Processed 40/120
Processed 41/120
Processed 42/120
Processed 43/120
Processed 44/120
Processed 45/120
Processed 46/120
Processed 47/120
Processed 48/120
Processed 49/120
Processed 50/120
Processed 51/120
Processed 52/120
Processed 53/120
Processed 54/120
Processed 55/120
Processed 56/120
Processed 57/120
Processed 58/120
Processed 59/120
Proces

In [192]:
enriched_rag_val_df = pd.DataFrame(
    enriched_rag_val_results
)

enriched_rag_cm = get_confusion_matrix(
    enriched_rag_val_df["actual"],
    enriched_rag_val_df["prediction"]
)

enriched_rag_metrics = evaluate_predictions(
    enriched_rag_val_df["actual"],
    enriched_rag_val_df["prediction"]
)

print("Confusion matrix:")
print(enriched_rag_cm)

print("\nMetrics:")
print(enriched_rag_metrics)

Confusion matrix:
[[24 21]
 [29 46]]

Metrics:
{'precision': 0.6865671641791045, 'recall': 0.6133333333333333, 'f1': 0.647887323943662, 'f2': 0.6267029972752044}


In [193]:
tn, fp, fn, tp = enriched_rag_cm.ravel()

specificity = tn / (tn + fp)

print("Specificity:", specificity)

Specificity: 0.5333333333333333


In [194]:
from src.data.preprocessing import build_claim_dataset
from src.data.split_data import split_claim_data
from src.rag.prepare_documents import build_training_documents
from src.data.s3_io import upload_documents_to_s3
from src.config import S3_BUCKET

In [195]:
claims_processed = build_claim_dataset(df)

train_df, val_df, test_df = split_claim_data(
    claims_processed
)

training_documents = build_training_documents(
    train_df
)

print("Training documents:", len(training_documents))

Training documents: 560


In [196]:
upload_documents_to_s3(
    bucket=S3_BUCKET,
    prefix="rag/train",
    documents=training_documents
)